In [1]:
import pandas as pd
import json
import random
from helpers.generate_messages_with_context import message_to_string_format


In [2]:
CHAT_DATA_PATH = r"..\data\rag\private_filtered.csv"
df = pd.read_csv(CHAT_DATA_PATH)

In [3]:
def select_messages_with_context_and_response(df, n=5, context_window=3, 
                                              author_col='Author', 
                                              content_col='Content',
                                              random_seed=None):
    """
    Select n messages where:
    1. The message has a different author's response after it
    2. NEITHER current message NOR immediate response contains <URL> or <ATTACH>
    3. Include previous n messages as context (always context_window count)
    4. Include the different author's response
    
    Args:
        df: DataFrame containing chat messages
        n: number of main messages to select
        context_window: number of previous messages to include as context
        author_col: column name for author/username
        content_col: column name for message content
        random_seed: random seed for reproducibility
    """
    if random_seed is not None:
        random.seed(random_seed)
    
    # Ensure we have the required columns
    if author_col not in df.columns:
        raise ValueError(f"Author column '{author_col}' not found. Available: {list(df.columns)}")
    
    if content_col not in df.columns:
        raise ValueError(f"Content column '{content_col}' not found. Available: {list(df.columns)}")
    
    # Find indices where:
    # 1. Current author is different from next author
    # 2. Current message does NOT contain <URL> or <ATTACH>
    # 3. Next message (response) does NOT contain <URL> or <ATTACH>
    valid_indices = []
    for i in range(context_window, len(df) - 1):  # -1 because we need next message
        current_author = str(df.iloc[i][author_col])
        next_author = str(df.iloc[i + 1][author_col])
        
        # Get message contents
        current_content = str(df.iloc[i][content_col])
        next_content = str(df.iloc[i + 1][content_col])
        
        # Check conditions
        has_valid_authors = (current_author and next_author and 
                            current_author != next_author)
        
        current_clean = ("<URL>" not in current_content and 
                        "<ATTACH>" not in current_content)
        
        response_clean = ("<URL>" not in next_content and 
                         "<ATTACH>" not in next_content)
        
        if has_valid_authors and current_clean and response_clean:
            valid_indices.append(i)
    
    if len(valid_indices) < n:
        raise ValueError(f"Not enough messages with clean different author responses. "
                        f"Found {len(valid_indices)}, need {n}. "
                        f"(Current or response messages with <URL> or <ATTACH> are excluded)")
    
    # Randomly select n unique indices
    selected_indices = random.sample(valid_indices, n)
    selected_indices.sort()
    
    results = []
    
    for idx in selected_indices:
        # Get context messages (previous n messages) - ALL messages included
        start_idx = max(0, idx - context_window)
        context_messages = df.iloc[start_idx:idx].to_dict('records')
        
        # Get current message (already filtered to be clean)
        current_message = df.iloc[idx].to_dict()
        current_author = str(current_message.get(author_col, ""))
        
        # Get response message (already filtered to be clean)
        response_message = df.iloc[idx + 1].to_dict()
        response_author = str(response_message.get(author_col, ""))
        
        # Get next few messages from the responding author (NO FILTERING here)
        additional_responses = []
        next_idx = idx + 2
        max_additional = 2  # Maximum additional responses to include
        count = 0
        
        while (count < max_additional and next_idx < len(df) and 
               str(df.iloc[next_idx][author_col]) == response_author):
            additional_responses.append(df.iloc[next_idx].to_dict())
            next_idx += 1
            count += 1
        
        result_entry = {
            "message_id": int(idx),
            "current_author": current_author,
            "response_author": response_author,
            "context_messages": context_messages,
            "context_count": len(context_messages),  # This will ALWAYS be context_window
            "current_message": current_message,
            "immediate_response": response_message,
            "additional_responses": additional_responses if additional_responses else None,
            "total_response_messages": 1 + len(additional_responses)
        }
        
        # Add content previews
        current_content = str(current_message.get(content_col, ""))
        response_content = str(response_message.get(content_col, ""))
        
        result_entry["current_preview"] = current_content[:100] + "..." if len(current_content) > 100 else current_content
        result_entry["response_preview"] = response_content[:100] + "..." if len(response_content) > 100 else response_content
        
        # Add flags to show what was filtered
        result_entry["current_has_url_or_attach"] = False  # Always False since we filtered
        result_entry["response_has_url_or_attach"] = False  # Always False since we filtered
        
        results.append(result_entry)
    
    return results

In [10]:
n_messages = 50
context_size = 10

selected_data = select_messages_with_context_and_response(
    df=df,
    n=n_messages,
    context_window=context_size,
    random_seed=45
)

# Save to JSON
output_path = r"..\output\rag\selected_messages_with_context.json"
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump({
        "metadata": {
            "total_messages_selected": n_messages,
            "context_window_size": context_size,
            "total_entries_in_dataset": len(df)
        },
        "selected_conversations": selected_data
    }, f, indent=2, ensure_ascii=False)

print(f"Successfully saved {n_messages} messages with context to {output_path}")
print(f"Each message has {context_size} previous messages as context")

Successfully saved 50 messages with context to ..\output\rag\selected_messages_with_context.json
Each message has 10 previous messages as context


In [ ]:
with open(output_path, 'r') as f:
    data = json.load(f)
# message_to_string_format(data, include_immediate_response=True)

for entry in data['selected_conversations']:
    # print(entry["current_message"]["Content"])
    print(entry["immediate_response"]["Content"])